# Mango Fruit V4.2.0: Combined-model threshold fitting

**Experimental candidate only. This clean notebook has no results and does not change app model files.**

First complete the matching Mango Fruit V4.2.0 training notebook. Its training run contains one Combined model: the existing geometric/brightness augmentation plus reviewed background replacement and local light/shadow patches. Training notebook also contains figure sections; run its post-threshold figures only after this notebook finishes.


## How to use this notebook

Run Steps 1–7 in order. Read each step's inputs, outputs, and checks. The two organs get separate calibrated weights, class orders, gate selections, and reports. Paste the exact completed organ training run path in Step 1.

Confidence stays at 0.70. Fit margin and entropy on **this organ's mango validation photos plus old OOD tune photos only**. Freeze selected gate before any mango test, old OOD report, or reused new OOD report inference. No test or report score can guide fitting. If no gate preserves 95% validation mango retention, report confidence-only reference and do not export a fitted candidate.

Old mango test and OOD images influenced prior model work. The 32 newer OOD photos were already reported in V4.1/V4.1.1. All are exploratory here, not independent field evidence. Neither model has an OOD output class.


## Step 1: Configure paths and runtime

**Reads:** Completed Mango Fruit training run; same old/new OOD Drive folders used in V4.1.1; and V4.1.1 `evaluation_inputs.json`, which records their image hashes and old tune/report assignment. **Saves:** Settings in memory only. Paste the complete run folder into `TRAINING_RUN`; verify `REFERENCE_V411_INPUTS` points to the saved V4.1.1 threshold run. If that reference file was moved, edit its path. Add OOD exclusions only for reviewed content reasons, never to improve scores.


### Setup cell

Mount Drive in a fresh Colab runtime. Source OOD folders are `/content/drive/MyDrive/mango_ood_test` and `/content/drive/MyDrive/mango_ood_test_new_photos`. V4.1.1 `evaluation_inputs.json` is required to prove the same files and old tune/report membership are reused. The second collection is report-only and already reused.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import csv, gc, hashlib, json, math, random, shutil, sys, uuid, zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
import numpy as np
import tensorflow as tf
from PIL import Image, ImageOps, ImageDraw
from scipy.fftpack import dct

SEED = 42
ARMS = ['combined']  # One model trained with common, background, and local-light augmentation.
CLASS_NAMES = ['bacterial_black_spot', 'healthy_fruit', 'mango_scab']  # Exact export order; never reorder after training.
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CONFIDENCE_THRESHOLD = 0.70
MIN_ID_RETENTION = 0.95  # Fraction of validation mango photos accepted, whether correct or wrong.
GRID = [i / 100 for i in range(101)]
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp'}
PREPROCESSING = 'TensorFlow RGB decode (Pillow RGB with EXIF transpose for WebP); bilinear 224x224, no antialias; float32 0-255; model rescales internally'
CONFIDENCE_ONLY = {'confidenceThreshold': 0.70, 'marginThreshold': 0.0, 'entropyThreshold': 1.0}

ORGAN = 'mangofruit'
EXPERIMENT_ROOT = Path('/content/drive/MyDrive/mangoleaf_experiments/v4.2.0')
TRAINING_RUN = EXPERIMENT_ROOT / 'training' / ORGAN / 'v4-2-0-mangofruit-training-20260928T164258Z-a4e1abae'
REFERENCE_V411_INPUTS = Path('/content/drive/MyDrive/mangoleaf_experiments/v4.1.1/threshold_fitting/v4-1-1-thresholds-20260928T060638Z-a3a7418c/evaluation_inputs.json')
# If the saved V4.1.1 run moved, paste its evaluation_inputs.json path above.
OLD_OOD_DIR = Path('/content/drive/MyDrive/mango_ood_test')
NEW_OOD_DIR = Path('/content/drive/MyDrive/mango_ood_test_new_photos')
OUTPUT_ROOT = EXPERIMENT_ROOT / 'threshold_fitting' / ORGAN
OLD_OOD_EXCLUDE = []  # Reviewed exact relative paths; changes to historical old split are recorded.
NEW_OOD_EXCLUDE = []  # Reviewed exact relative paths, e.g. images containing mango.
EXCLUSION_REASONS = {}  # Key: 'old/path.jpg' or 'new/path.jpg'; value: content-based reason.

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.keras.mixed_precision.set_global_policy('float32')
RUNTIME = {'python': sys.version, 'tensorflow': tf.__version__,
           'keras': tf.keras.__version__, 'numpy': np.__version__, 'pillow': Image.__version__}
print('Runtime:', RUNTIME)
print('Training run:', TRAINING_RUN)
print('Threshold output root:', OUTPUT_ROOT)

Mounted at /content/drive
Runtime: {'python': '3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]', 'tensorflow': '2.20.0', 'keras': '3.13.2', 'numpy': '2.1.3', 'pillow': '11.3.0'}
Training run: /content/drive/MyDrive/mangoleaf_experiments/v4.2.0/training/mangofruit/v4-2-0-mangofruit-training-20260928T164258Z-a4e1abae
Threshold output root: /content/drive/MyDrive/mangoleaf_experiments/v4.2.0/threshold_fitting/mangofruit


### File and split checks

Helpers verify SHA-256 hashes, byte sizes, and safe ZIP paths before inference. The saved split must contain only the matching organ's train, validation, and test photos; their existing V4.1.1 membership is preserved by training.


In [2]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def object_hash(value):
    text = json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=True)
    return hashlib.sha256(text.encode()).hexdigest()


def write_json_once(path, value):
    with Path(path).open('x', encoding='utf-8') as handle:
        json.dump(value, handle, indent=2, allow_nan=False)
        handle.write('\n')


def list_images(folder):
    folder = Path(folder)
    if not folder.is_dir():
        raise FileNotFoundError(f'Image folder missing: {folder}')
    paths = sorted(p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    if not paths:
        raise ValueError(f'No supported images in {folder}')
    return paths


def verify_records_unchanged(rows):
    for row in rows:
        if sha256_file(row['path']) != row['sha256']:
            raise RuntimeError('Image changed: ' + row['path'])


def inside(root, relative):
    relative = PurePosixPath(relative)
    if relative.is_absolute() or '..' in relative.parts or '\\' in str(relative):
        raise ValueError('Unsafe relative path: ' + str(relative))
    target = (Path(root) / str(relative)).resolve()
    if not target.is_relative_to(Path(root).resolve()) or target == Path(root).resolve():
        raise ValueError('Path must stay inside its output folder.')
    return target


def extract_dataset(archive, rows, destination):
    # Extract only recorded images, then verify their contents before inference.
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=False)
    expected = {r['relativePath']: r for r in rows}
    if len(expected) != len(rows):
        raise ValueError('The image manifest repeats a destination path.')
    with zipfile.ZipFile(archive) as bundle:
        names = bundle.namelist()
        if len(names) != len(set(names)) or set(names) != set(expected):
            raise ValueError('Dataset ZIP entries differ from the image manifest.')
        for name, row in expected.items():
            target = inside(destination, name)
            if bundle.getinfo(name).file_size != row['bytes']:
                raise ValueError('Image size changed in dataset ZIP: ' + name)
            target.parent.mkdir(parents=True, exist_ok=True)
            with bundle.open(name) as source, target.open('xb') as output:
                shutil.copyfileobj(source, output)
            if sha256_file(target) != row['sha256']:
                raise ValueError('Image contents changed in dataset ZIP: ' + name)
    return [{**r, 'path': str(inside(destination, r['relativePath']))} for r in rows]


def verify_artifacts(folder, artifacts):
    for relative, expected in artifacts.items():
        path = inside(folder, relative)
        if not path.is_file() or sha256_file(path) != expected:
            raise RuntimeError('Missing or changed saved artifact: ' + relative)

### Model input and prediction helpers

Read 224 × 224 RGB photos exactly as training did. The calibrated model must return 3 probabilities per photo in frozen class order. Validation predictions will be compared with training's saved `.npy` values. This is a Keras check; browser TensorFlow.js parity needs separate testing.


In [3]:
def load_pixels(path):
    # JPEG/PNG/GIF/BMP use TensorFlow decode; WebP uses Pillow RGB. Keep 0-255 pixels.
    if Path(path).suffix.lower() == '.webp':
        # TensorFlow decode_image cannot decode WebP; match the training notebook's RGB branch.
        with Image.open(path) as picture:
            pixels = np.asarray(ImageOps.exif_transpose(picture).convert('RGB'), dtype=np.float32)
    else:
        pixels = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        pixels.set_shape((None, None, 3))
    return tf.image.resize(pixels, IMAGE_SIZE, method='bilinear',
                           preserve_aspect_ratio=False, antialias=False)


def predict_records(model, rows):
    output = []
    for start in range(0, len(rows), BATCH_SIZE):
        batch = rows[start:start + BATCH_SIZE]
        verify_records_unchanged(batch)
        pixels = tf.stack([load_pixels(r['path']) for r in batch])
        values = model(pixels, training=False).numpy()
        if values.shape != (len(batch), len(CLASS_NAMES)) or not np.isfinite(values).all():
            raise ValueError('Model output count or finite-probability check failed for this organ.')
        output.extend(values.tolist())
    if not output:
        raise ValueError('Cannot predict an empty image collection.')
    return np.asarray(output, dtype=np.float32)


def weight_digest(model):
    digest = hashlib.sha256()
    for variable in model.weights:
        values = variable.numpy()
        digest.update(str(values.shape).encode())
        digest.update(str(values.dtype).encode())
        digest.update(values.tobytes())
    return digest.hexdigest()


def print_class_probabilities(probabilities, title):
    # Sort label-score pairs for display; keep the model's class order unchanged.
    ranked = sorted(zip(CLASS_NAMES, map(float, probabilities), strict=True),
                    key=lambda item: item[1], reverse=True)
    print('\n' + title)
    for label, probability in ranked:
        print(f"{label.replace('_', ' ').title():<24} {probability * 100:6.2f}%")

### TensorFlow.js candidate checks

Before packaging, verify browser graph input/output shapes, finite unquantized weights, and every weight shard. Threshold fitting does not alter graph weights.


In [4]:
def verify_tfjs_export(folder):
    folder = Path(folder)
    graph = json.loads((folder / 'model.json').read_text(encoding='utf-8'))
    if graph.get('format') != 'graph-model':
        raise ValueError('Expected a TensorFlow.js Graph Model.')
    signature = graph.get('signature', {})
    outputs = list(signature.get('outputs', {}).values())
    inputs = list(signature.get('inputs', {}).values())
    dims = lambda tensor: [int(d['size']) for d in tensor['tensorShape']['dim']]
    if len(outputs) != 1 or dims(outputs[0])[-1:] != [len(CLASS_NAMES)]:
        raise ValueError('Export output count differs from this organ class list.')
    if len(inputs) != 1 or dims(inputs[0])[1:] != [224, 224, 3]:
        raise ValueError('Export input shape differs from the training plan.')
    dtypes = {'float32': '<f4', 'int32': '<i4', 'bool': '?'}
    used = set()
    for group in graph['weightsManifest']:
        parts = []
        for name in group['paths']:
            if name in used:
                raise ValueError('Repeated weight shard: ' + name)
            used.add(name)
            parts.append(inside(folder, name).read_bytes())
        buffer, offset = b''.join(parts), 0
        for weight in group['weights']:
            if 'quantization' in weight or weight['dtype'] not in dtypes:
                raise ValueError('This experiment requires unquantized float32 weights.')
            dtype = np.dtype(dtypes[weight['dtype']])
            count = math.prod(weight['shape'])
            size = count * dtype.itemsize
            if offset + size > len(buffer):
                raise ValueError('Truncated weight shard.')
            values = np.frombuffer(buffer[offset:offset + size], dtype=dtype)
            if not np.isfinite(values).all():
                raise ValueError('Non-finite values found in the exported weights.')
            offset += size
        if offset != len(buffer):
            raise ValueError('Weight manifest and shard sizes disagree.')
    if not used:
        raise ValueError('No exported weight shards.')
    return ['model.json'] + sorted(used)

### Rejection gate calculations

Confidence is highest class score. Margin is top score minus second score. Normalized entropy measures spread across all 3 class scores using `ln(3)`, yielding a 0–1 scale. Accept only when confidence and margin are high enough and entropy is low enough. Fitted thresholds remain organ-specific even though entropy uses the same scale.


In [5]:
def probability_stats(probabilities):
    stats = []
    for row in probabilities:
        values = [float(p) for p in row]
        if len(values) != len(CLASS_NAMES) or any(
                not math.isfinite(p) or p < 0 or p > 1 for p in values):
            raise ValueError('Expected one finite probability per organ class in [0, 1].')
        if not math.isclose(sum(values), 1.0, abs_tol=1e-4):
            raise ValueError('Expected softmax probabilities summing to one.')
        ordered = sorted(values, reverse=True)
        stats.append({'confidence': ordered[0], 'margin': ordered[0] - ordered[1],
                      'entropy': -sum(p * math.log(p) for p in values if p > 0)
                                 / math.log(len(values)),
                      'predictedIndex': values.index(ordered[0]),
                      # Keep full precision in CSVs; only preview text is rounded.
                      **{f'probability_{label}': p for label, p in zip(CLASS_NAMES, values)}})
    return stats


def accepted(stat, thresholds):
    return (stat['confidence'] >= thresholds['confidenceThreshold']
            and stat['margin'] >= thresholds['marginThreshold']
            and stat['entropy'] <= thresholds['entropyThreshold'])


def select_thresholds(val_stats, ood_tune_stats, grid, confidence, min_retention):
    # Only tuning data enters this function. Counts avoid rounded-score ties.
    if not val_stats or not ood_tune_stats:
        raise ValueError('Both tuning subsets must contain predictions.')
    if not 0 < min_retention <= 1 or not 0 <= confidence <= 1:
        raise ValueError('Invalid confidence or retention target.')
    if not grid or any(not math.isfinite(x) or not 0 <= x <= 1 for x in grid):
        raise ValueError('Grid must contain finite values between zero and one.')
    required = math.ceil(min_retention * len(val_stats) - 1e-9)
    best, best_key, rows = None, None, []
    for margin in grid:
        val_eligible = [s for s in val_stats if s['confidence'] >= confidence and s['margin'] >= margin]
        ood_eligible = [s for s in ood_tune_stats if s['confidence'] >= confidence and s['margin'] >= margin]
        for entropy in grid:
            kept = sum(s['entropy'] <= entropy for s in val_eligible)
            rejected = len(ood_tune_stats) - sum(s['entropy'] <= entropy for s in ood_eligible)
            feasible = kept >= required
            rows.append({'margin': margin, 'entropy': entropy, 'valAccepted': kept,
                         'valTotal': len(val_stats), 'oodRejected': rejected,
                         'oodTotal': len(ood_tune_stats), 'meetsRetentionTarget': feasible})
            # Ties: more mango retention, then lower margin, then higher entropy.
            key = (rejected, kept, -margin, entropy)
            if feasible and (best_key is None or key > best_key):
                best_key = key
                best = {'confidenceThreshold': confidence, 'marginThreshold': margin,
                        'entropyThreshold': entropy}
    return best, rows

### Per-image and summary helpers

Every row records full-precision scores, predicted class, accepted/uncertain decision, and failed gates. `acceptedWrong` is a wrong diagnosis the app would display. Validation retention counts all accepted mango photos, including wrong ones; it is not accuracy.


In [6]:
def scored_rows(records, stats, thresholds):
    if len(records) != len(stats):
        raise ValueError('Image and prediction counts differ.')
    rows = []
    for record, stat in zip(records, stats):
        keep = accepted(stat, thresholds)
        prediction = CLASS_NAMES[stat['predictedIndex']]
        reasons = []
        if stat['confidence'] < thresholds['confidenceThreshold']: reasons.append('confidence')
        if stat['margin'] < thresholds['marginThreshold']: reasons.append('margin')
        if stat['entropy'] > thresholds['entropyThreshold']: reasons.append('entropy')
        rows.append({**record, **stat, 'predictedClass': prediction, 'accepted': keep,
                     'returnedClass': prediction if keep else 'uncertain',
                     'rejectionReasons': ','.join(reasons),
                     'acceptedCorrect': keep and prediction == record['label']})
    return rows


def summarize(rows):
    if not rows:
        raise ValueError('Cannot report an empty subset.')
    count = len(rows)
    kept = sum(row['accepted'] for row in rows)
    report = {'total': count, 'accepted': kept, 'rejected': count - kept,
              'retention': kept / count, 'rejectionRate': (count - kept) / count}
    if all(row['label'] == 'OOD' for row in rows):
        report['falseAcceptRate'] = kept / count
        return report
    if any(row['label'] == 'OOD' for row in rows):
        raise ValueError('Report mango and OOD subsets separately.')
    correct = sum(row['acceptedCorrect'] for row in rows)
    report.update({'rawAccuracy': sum(row['predictedClass'] == row['label'] for row in rows) / count,
                   'acceptedCorrect': correct, 'acceptedWrong': kept - correct,
                   'acceptedAccuracy': correct / kept if kept else None,
                   'acceptedErrorsPerInput': (kept - correct) / count})
    per_class = {}
    confusion = [[0] * (len(CLASS_NAMES) + 1) for _ in CLASS_NAMES]
    for row in rows:
        column = row['predictedIndex'] if row['accepted'] else len(CLASS_NAMES)
        confusion[CLASS_NAMES.index(row['label'])][column] += 1
    for label in CLASS_NAMES:
        group = [row for row in rows if row['label'] == label]
        class_kept = sum(row['accepted'] for row in group)
        class_correct = sum(row['acceptedCorrect'] for row in group)
        per_class[label] = {'total': len(group), 'acceptedCorrect': class_correct,
                            'acceptedWrong': class_kept - class_correct,
                            'rejected': len(group) - class_kept,
                            'retention': class_kept / len(group) if group else None}
    healthy = {'healthy_leaf', 'healthy_fruit'}
    report['healthyErrorCounts'] = {
        'healthyAcceptedAsDisease': sum(r['accepted'] and r['label'] in healthy and r['predictedClass'] not in healthy for r in rows),
        'diseaseAcceptedAsHealthy': sum(r['accepted'] and r['label'] not in healthy and r['predictedClass'] in healthy for r in rows),
        'healthyOrganConfusion': sum(r['accepted'] and r['label'] in healthy and r['predictedClass'] in healthy and r['label'] != r['predictedClass'] for r in rows)}
    report['perClass'] = per_class
    report['confusion'] = {'rows': CLASS_NAMES, 'columns': CLASS_NAMES + ['uncertain'], 'counts': confusion}
    return report

### Write-once evidence helpers

Saved CSV files cannot be overwritten. Before held-out evaluation, read the frozen gate file and compare its SHA-256 hash. To alter any setting after selection, start a new run.


In [7]:
def write_csv_once(path, rows):
    if not rows:
        raise ValueError('Cannot write an empty report.')
    with Path(path).open('x', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def read_frozen_selection(path, expected_sha256):
    if sha256_file(path) != expected_sha256:
        raise RuntimeError('Saved selection changed. Start a new run; do not retune during evaluation.')
    return json.loads(Path(path).read_text(encoding='utf-8'))

### OOD inventory helpers

Keep the old OOD `tune/` and `report/` folders if present. For a flat historical folder, reproduce its sorted-path `RandomState(42)` 50/50 split. All new OOD photos are report-only. The notebook rejects unsupported image formats and symlink escapes.


In [8]:
def reviewed_paths(root):
    """Reject symlink escapes and unsupported folder layouts before any predictions."""
    root = Path(root).resolve()
    unsupported = sorted(str(p.relative_to(root)) for p in root.rglob('*') if p.is_file()
                         and p.suffix.lower() in {'.heic', '.avif', '.tif', '.tiff'})
    if unsupported:
        raise ValueError('Convert unsupported image format to JPEG or PNG, then start a new run: '
                         + ', '.join(unsupported[:5]))
    paths = list_images(root)
    if any(not p.resolve().is_relative_to(root) for p in paths):
        raise ValueError('An image symlink leaves its source folder.')
    return paths


def image_record(root, path, subset):
    with Image.open(path) as picture:
        picture.verify()
    return {'subset': subset, 'label': 'OOD', 'path': str(path.resolve()),
            'relativePath': path.relative_to(root).as_posix(),
            'sha256': sha256_file(path), 'bytes': path.stat().st_size}


def prepare_old_ood(root, excluded):
    root = Path(root).resolve()
    paths = reviewed_paths(root)
    all_names = {p.relative_to(root).as_posix() for p in paths}
    if len(excluded) != len(set(excluded)) or not set(excluded) <= all_names:
        raise ValueError('OLD_OOD_EXCLUDE repeats or names a missing image.')
    tune_dir, report_dir = root / 'tune', root / 'report'
    if tune_dir.is_dir() != report_dir.is_dir():
        raise ValueError('Old OOD needs both tune/ and report/, or neither.')
    if tune_dir.is_dir():
        if any(p.relative_to(root).parts[0] not in ('tune', 'report') for p in paths):
            raise ValueError('Old OOD photos outside tune/report would be ignored.')
        assigned = [('ood_tune', p) if p.is_relative_to(tune_dir) else ('ood_report', p)
                    for p in paths]
        method = 'existing tune/ and report/ directories'
    else:
        if any(p.parent != root for p in paths):
            raise ValueError('Old OOD must be flat or have explicit tune/report folders.')
        # Match V4 historical assignment: sorted full path, RandomState(42), half for tune.
        paths = sorted(paths)
        order = np.random.RandomState(SEED).permutation(len(paths))
        half = len(paths) // 2
        tune = {int(i) for i in order[:half]}
        assigned = [('ood_tune' if i in tune else 'ood_report', p) for i, p in enumerate(paths)]
        method = 'historical V4 flat-folder split: sorted paths, RandomState(42), 50/50'
    rows = [image_record(root, p, subset) for subset, p in assigned
            if p.relative_to(root).as_posix() not in excluded]
    if not any(r['subset'] == 'ood_tune' for r in rows) or not any(r['subset'] == 'ood_report' for r in rows):
        raise ValueError('Old OOD tune and report both need retained images.')
    return rows, method


def prepare_new_ood(root, excluded):
    root = Path(root).resolve()
    paths = reviewed_paths(root)
    all_names = {p.relative_to(root).as_posix() for p in paths}
    if len(excluded) != len(set(excluded)) or not set(excluded) <= all_names:
        raise ValueError('NEW_OOD_EXCLUDE repeats or names a missing image.')
    rows = [image_record(root, p, 'ood_new_report') for p in paths
            if p.relative_to(root).as_posix() not in excluded]
    if not rows:
        raise ValueError('New OOD report needs at least one eligible image.')
    return rows, paths

### Overlap audit and review sheets

Exact image duplicates across collections stop the run. Perceptual-hash matches are visual review flags, not automatic proof. Contact sheets and index CSVs let reviewer see every new OOD photo and map it to its saved hash/path.


In [9]:
def perceptual_hashes(path):
    """Compare all right-angle rotations and mirrors; pHash flags need human review."""
    with Image.open(path) as picture:
        base = ImageOps.exif_transpose(picture).convert('L')
    rotations = [base.rotate(angle, expand=True) for angle in (0, 90, 180, 270)]
    views = rotations + [ImageOps.mirror(view) for view in rotations]
    hashes = []
    for view in views:
        pixels = np.asarray(view.resize((32, 32)), dtype=np.float32)
        frequency = dct(dct(pixels, axis=0, norm='ortho'), axis=1, norm='ortho')[:8, :8]
        median = np.median(frequency.flatten()[1:])
        hashes.append(sum(int(value > median) << i for i, value in enumerate(frequency.flatten())))
    return hashes


def audit_duplicate_pairs(rows):
    """Exact matches stop; near matches need individual visual review and written reasons."""
    seen = {}
    for row in rows:
        if row['sha256'] in seen:
            first = seen[row['sha256']]
            raise ValueError('Exact duplicate: ' + first['subset'] + '/' + first['relativePath']
                             + ' and ' + row['subset'] + '/' + row['relativePath'])
        seen[row['sha256']] = row
    hashes = [(row, perceptual_hashes(row['path'])) for row in rows]
    flagged = []
    for i, (left, left_hashes) in enumerate(hashes):
        for right, right_hashes in hashes[i + 1:]:
            # Only cross-collection or cross-subset comparisons can leak into an evaluation.
            if left['subset'] == right['subset'] and left['subset'] in ('train', 'val', 'test'):
                continue
            if not (left['label'] == 'OOD' or right['label'] == 'OOD'):
                continue
            # Dihedral transformations of one image cover every relative rotation/mirror.
            distance = min((candidate ^ right_hashes[0]).bit_count() for candidate in left_hashes)
            if distance > 4:
                continue
            left_id = left['subset'] + '/' + left['relativePath'] + '/' + left['sha256']
            right_id = right['subset'] + '/' + right['relativePath'] + '/' + right['sha256']
            pair_id = hashlib.sha256('|'.join(sorted((left_id, right_id))).encode()).hexdigest()[:16]
            flagged.append({'pairId': pair_id, 'hammingDistance': distance,
                            'leftSubset': left['subset'], 'leftPath': left['relativePath'],
                            'leftSha256': left['sha256'], 'rightSubset': right['subset'],
                            'rightPath': right['relativePath'], 'rightSha256': right['sha256']})
    return flagged


def make_contact_sheets(rows, folder, prefix):
    """Every OOD photo gets an indexed thumbnail; mapping is saved as CSV."""
    folder.mkdir(parents=True, exist_ok=True)
    index = []
    for page_start in range(0, len(rows), 20):
        sheet = Image.new('RGB', (1000, 800), 'white')
        draw = ImageDraw.Draw(sheet)
        for offset, row in enumerate(rows[page_start:page_start + 20]):
            number = page_start + offset
            x, y = (offset % 5) * 200, (offset // 5) * 200
            with Image.open(row['path']) as picture:
                image = ImageOps.exif_transpose(picture).convert('RGB')
                image.thumbnail((190, 155))
                sheet.paste(image, (x + (200 - image.width) // 2, y))
            draw.text((x + 5, y + 158), f'{number:03d} {row["relativePath"][-24:]}', fill='black')
            index.append({'index': number, 'subset': row['subset'], 'relativePath': row['relativePath'],
                          'sha256': row['sha256']})
        sheet.save(folder / f'{prefix}_{page_start // 20:03d}.jpg', quality=88)
    write_csv_once(folder / f'{prefix}_index.csv', index)


def make_near_pair_sheets(pairs, rows, folder):
    """Place flagged images side by side so reviewers can resolve pair IDs visually."""
    locations = {(row['subset'], row['relativePath']): row['path'] for row in rows}
    for page_start in range(0, len(pairs), 5):
        sheet = Image.new('RGB', (1000, 1000), 'white')
        draw = ImageDraw.Draw(sheet)
        for offset, pair in enumerate(pairs[page_start:page_start + 5]):
            y = offset * 200
            for side, x in [('left', 0), ('right', 500)]:
                key = (pair[side + 'Subset'], pair[side + 'Path'])
                with Image.open(locations[key]) as picture:
                    image = ImageOps.exif_transpose(picture).convert('RGB')
                    image.thumbnail((485, 155))
                    sheet.paste(image, (x + (500 - image.width) // 2, y))
                draw.text((x + 5, y + 158), f"{pair['pairId']} {side}: {key[0]}/{key[1][-32:]}", fill='black')
        sheet.save(folder / f'near_pairs_{page_start // 5:03d}.jpg', quality=88)

### Model reload and frozen-input checks

Recheck calibrated Keras model shape, V4.1.1 OOD reference, and all relevant hashes before fitting, evaluating, and packaging. This prevents a mixed model/photo/threshold result.


In [10]:
def verify_all_records_unchanged(rows):
    verify_records_unchanged(rows)
    for row in rows:
        if Path(row['path']).stat().st_size != row['bytes']:
            raise RuntimeError('Image size changed after audit: ' + row['path'])


def load_arm(arm):
    info = arm_infos[arm]
    if sha256_file(info['modelPath']) != info['modelSha256']:
        raise RuntimeError('Model changed after audit: ' + arm)
    if sha256_file(info['infoPath']) != info['infoSha256']:
        raise RuntimeError('Calibration metadata changed after audit: ' + arm)
    model = tf.keras.models.load_model(info['modelPath'], compile=False, safe_mode=True)
    if tuple(model.input_shape[1:]) != (224, 224, 3) or tuple(model.output_shape[1:]) != (len(CLASS_NAMES),):
        raise ValueError('Unexpected model input/output shape: ' + arm)
    return model


def verify_frozen_inputs():
    if sha256_file(REFERENCE_V411_INPUTS) != REFERENCE_V411_INPUTS_SHA256:
        raise RuntimeError('V4.1.1 OOD reference changed after preflight.')
    if sha256_file(TRAINING_RUN / 'training_complete.json') != COMPLETE_SHA256:
        raise RuntimeError('Training completion record changed.')
    if sha256_file(TRAINING_RUN / 'input_manifest.json') != TRAIN_MANIFEST_SHA256:
        raise RuntimeError('Training input manifest changed.')
    if sha256_file(TRAINING_RUN / 'dataset_split.zip') != complete['datasetSplitSha256']:
        raise RuntimeError('Saved split changed.')
    if sha256_file(INPUT_PATH) != INPUT_SHA256:
        raise RuntimeError('Evaluation input manifest changed.')
    verify_all_records_unchanged(all_records)
    for arm in ARMS:
        info = arm_infos[arm]
        if sha256_file(info['modelPath']) != info['modelSha256']:
            raise RuntimeError('Model weights changed: ' + arm)
        if sha256_file(info['infoPath']) != info['infoSha256']:
            raise RuntimeError('Model metadata changed: ' + arm)

## Step 2: Audit inputs before inference

**Reads:** Completed training bundle, model hashes, split ZIP, both OOD folders, and V4.1.1 `evaluation_inputs.json`. **Saves:** `audit/preflight.json`, indexed OOD contact sheets, near-duplicate flags, and temporary extracted hash-checked mango photos. **Check:** Correct organ/classes, same old OOD tune/report membership and same OOD image bytes as V4.1.1, complete training record, no earlier test prediction. Review sheets before Step 3.


In [11]:
COMPLETE_PATH = TRAINING_RUN / 'training_complete.json'
if not COMPLETE_PATH.is_file():
    raise FileNotFoundError('Paste a completed V4.2.0 organ training run path before continuing.')
complete = json.loads(COMPLETE_PATH.read_text(encoding='utf-8'))
COMPLETE_SHA256 = sha256_file(COMPLETE_PATH)
TRAIN_MANIFEST_PATH = TRAINING_RUN / 'input_manifest.json'
TRAIN_MANIFEST_SHA256 = sha256_file(TRAIN_MANIFEST_PATH)
if complete['inputManifestSha256'] != TRAIN_MANIFEST_SHA256 or complete['testPredictionsRun'] is not False:
    raise ValueError('Training manifest changed or test predictions already ran during training.')
if (set(complete['arms']) != set(ARMS) or complete['classes'] != CLASS_NAMES or
        complete.get('armOrder') != ARMS or complete.get('organ') != ORGAN):
    raise ValueError('Completed run must have the selected organ class order and one Combined model.')
if sha256_file(TRAINING_RUN / 'dataset_split.zip') != complete['datasetSplitSha256']:
    raise ValueError('Saved dataset ZIP does not match the training completion record.')
verify_artifacts(TRAINING_RUN, complete['artifacts'])
manifest = json.loads(TRAIN_MANIFEST_PATH.read_text(encoding='utf-8'))
if manifest['classes'] != CLASS_NAMES or not manifest['images']:
    raise ValueError('Unexpected training split manifest.')
if set(r['subset'] for r in manifest['images']) != {'train', 'val', 'test'}:
    raise ValueError('Training manifest must list only train/val/test in dataset_split.zip.')
if any(r['label'] not in CLASS_NAMES for r in manifest['images']):
    raise ValueError('Unknown mango label in split manifest.')
if any(not Path(r['relativePath']).as_posix().startswith(r['subset'] + '/') for r in manifest['images']):
    raise ValueError('Split path and subset disagree.')

arm_infos = {}
for arm in ARMS:
    arm_dir = TRAINING_RUN / 'arms' / arm
    info_path = arm_dir / 'resume_info.json'
    model_path = arm_dir / 'model_calibrated.keras'
    prob_path = arm_dir / 'validation_probabilities.npy'
    info = json.loads(info_path.read_text(encoding='utf-8'))
    if (info['arm'] != arm or info['classes'] != CLASS_NAMES or
            info['seed'] != SEED or info['testPredictionsRun'] is not False or
            info['inputManifestSha256'] != TRAIN_MANIFEST_SHA256 or
            info['datasetSplitSha256'] != complete['datasetSplitSha256'] or
            info['validationImageCount'] != sum(r['subset'] == 'val' for r in manifest['images']) or
            info['modelSha256'] != sha256_file(model_path) or
            info['validationProbabilitiesSha256'] != sha256_file(prob_path) or
            info['temperatureFoldedIntoWeights'] is not True or
            info['preprocessing'] != PREPROCESSING or
            info['confidenceThreshold'] != CONFIDENCE_THRESHOLD or
            info['minimumValidationRetention'] != MIN_ID_RETENTION or
            not math.isfinite(float(info['temperature'])) or info['temperature'] <= 0 or
            info['initialWeightsSha256'] != complete['initialWeightsSha256']):
        raise ValueError('Incompatible saved calibration or training provenance: ' + arm)
    arm_complete = complete['arms'][arm]
    if {k: v for k, v in arm_complete.items() if k not in ('tfjsDirectory', 'tfjsArtifacts')} != info:
        raise ValueError('Completed arm metadata differs from its saved calibration record: ' + arm)
    if 'modelSha256' in arm_complete and arm_complete['modelSha256'] != info['modelSha256']:
        raise ValueError('Completion and arm model hashes disagree: ' + arm)
    if 'resumeInfoSha256' in arm_complete and arm_complete['resumeInfoSha256'] != sha256_file(info_path):
        raise ValueError('Completion and arm metadata hashes disagree: ' + arm)
    arm_infos[arm] = {**info, 'modelPath': str(model_path), 'infoPath': str(info_path),
                      'probPath': str(prob_path), 'infoSha256': sha256_file(info_path)}
if len({arm_infos[a]['modelVersion'] for a in ARMS}) != 1:
    raise ValueError('Proposed model label is inconsistent.')
if len({arm_infos[a]['planSha256'] for a in ARMS}) != 1 or         len({json.dumps(arm_infos[a]['classWeights'], sort_keys=True) for a in ARMS}) != 1:
    raise ValueError('Model plan or class weights disagree.')
MODEL_VERSION = arm_infos[ARMS[0]]['modelVersion']
if complete['modelVersion'] != MODEL_VERSION:
    raise ValueError('Training completion model version disagrees with saved weights.')

if not REFERENCE_V411_INPUTS.is_file():
    raise FileNotFoundError('V4.1.1 evaluation_inputs.json missing. Set REFERENCE_V411_INPUTS in Step 1 to the completed V4.1.1 threshold run.')
reference_v411 = json.loads(REFERENCE_V411_INPUTS.read_text(encoding='utf-8'))
REFERENCE_V411_INPUTS_SHA256 = sha256_file(REFERENCE_V411_INPUTS)
run_id = 'v4-2-0-mangofruit-thresholds-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
RUN_DIR = OUTPUT_ROOT / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
DATASET_DIR = Path('/content/mangofruit-threshold-v4-2-0') / run_id / 'dataset'
mango_records = extract_dataset(TRAINING_RUN / 'dataset_split.zip', manifest['images'], DATASET_DIR)
old_rows, old_split_method = prepare_old_ood(OLD_OOD_DIR, OLD_OOD_EXCLUDE)
new_rows, new_all_paths = prepare_new_ood(NEW_OOD_DIR, NEW_OOD_EXCLUDE)
def ood_identity(rows):
    return sorted((row['subset'], row['relativePath'], row['sha256']) for row in rows)
expected_old = [row for row in reference_v411['oldOod']
                if row['relativePath'] not in OLD_OOD_EXCLUDE]
expected_new = [row for row in reference_v411['newOod']
                if row['relativePath'] not in NEW_OOD_EXCLUDE]
if ood_identity(old_rows) != ood_identity(expected_old):
    raise ValueError('Old OOD file hash or tune/report membership differs from V4.1.1. Restore the same folder, or document a new experiment.')
if ood_identity(new_rows) != ood_identity(expected_new):
    raise ValueError('New OOD file hashes differ from V4.1.1. Restore the same 32-photo folder, or document a new experiment.')
if len(reference_v411['newOod']) != 32:
    raise ValueError('V4.1.1 OOD reference is not the expected 32-photo collection.')
background_rows = []
for item in manifest.get('backgroundImages', []):
    source = Path(item['path'])
    if not source.is_file() or sha256_file(source) != item['sha256'] or source.stat().st_size != item['bytes']:
        raise RuntimeError('Training background source changed: ' + str(source))
    background_rows.append({**item, 'subset': 'background_source', 'label': 'BACKGROUND'})
all_records = mango_records + background_rows + old_rows + new_rows
verify_all_records_unchanged(all_records)
near_pairs = audit_duplicate_pairs(all_records)
AUDIT_DIR = RUN_DIR / 'audit'
AUDIT_DIR.mkdir(exist_ok=False)
new_full_rows = [image_record(NEW_OOD_DIR.resolve(), p, 'ood_new_report') for p in new_all_paths]
make_contact_sheets(new_full_rows, AUDIT_DIR, 'new_ood')
make_near_pair_sheets(near_pairs, all_records, AUDIT_DIR)
make_contact_sheets(old_rows, AUDIT_DIR, 'old_ood')
if near_pairs:
    write_csv_once(AUDIT_DIR / 'near_duplicate_pairs.csv', near_pairs)
else:
    write_json_once(AUDIT_DIR / 'near_duplicate_pairs.json', [])
for prefix, excluded in [('old', OLD_OOD_EXCLUDE), ('new', NEW_OOD_EXCLUDE)]:
    for relative in excluded:
        reason = EXCLUSION_REASONS.get(prefix + '/' + relative, '').strip()
        if len(reason) < 12:
            raise ValueError('Give content-based exclusion reason for ' + prefix + '/' + relative)
counts = {subset: sum(r['subset'] == subset for r in all_records)
          for subset in ['train', 'val', 'test', 'background_source',
                         'ood_tune', 'ood_report', 'ood_new_report']}
write_json_once(AUDIT_DIR / 'preflight.json', {
    'trainingRun': str(TRAINING_RUN), 'trainingCompleteSha256': COMPLETE_SHA256,
    'trainingManifestSha256': TRAIN_MANIFEST_SHA256,
    'referenceV411EvaluationInputsSha256': REFERENCE_V411_INPUTS_SHA256,
    'oldOodSplitMethod': old_split_method,
    'oldOodExcluded': OLD_OOD_EXCLUDE, 'newOodExcluded': NEW_OOD_EXCLUDE,
    'exclusionReasons': EXCLUSION_REASONS, 'counts': counts,
    'newOodTotalUploaded': len(new_all_paths), 'newOodRetained': len(new_rows),
    'nearDuplicateFlagCount': len(near_pairs),
    'nearDuplicateThreshold': '64-bit DCT pHash, rotations/mirrors, Hamming distance <= 4',
    'runtime': RUNTIME, 'createdAt': datetime.now(timezone.utc).isoformat()})
print('Counts:', counts)
print('New OOD uploaded:', len(new_all_paths), 'retained:', len(new_rows))
print('Possible near-duplicate pairs requiring review:', len(near_pairs))
print('Contact sheets and index:', AUDIT_DIR)
print('Output run:', RUN_DIR)

Counts: {'train': 441, 'val': 94, 'test': 96, 'background_source': 62, 'ood_tune': 84, 'ood_report': 84, 'ood_new_report': 32}
New OOD uploaded: 32 retained: 32
Possible near-duplicate pairs requiring review: 0
Contact sheets and index: /content/drive/MyDrive/mangoleaf_experiments/v4.2.0/threshold_fitting/mangofruit/v4-2-0-mangofruit-thresholds-20260928T193804Z-fc01d0f7/audit
Output run: /content/drive/MyDrive/mangoleaf_experiments/v4.2.0/threshold_fitting/mangofruit/v4-2-0-mangofruit-thresholds-20260928T193804Z-fc01d0f7


### Check audit output

Inspect every new OOD sheet and each possible near duplicate. Confirm preflight counts and old OOD split method. If any photo is unsuitable, change only content-based exclusion lists with reasons and start a fresh threshold run.


## Step 3: Record manual scope review

**Reads:** Contact sheets and flagged pairs. **Saves:** `evaluation_inputs.json`, including reviewed hashes and notes. Mark scope reviewed only after checking every indexed new OOD image. Explain each flagged pair; exclude actual repeats before fitting.


In [12]:
NEW_OOD_SCOPE_REVIEWED = True  # Change only after inspecting every new_ood contact-sheet entry.
SCOPE_REVIEW_NOTES = 'Reviewed all the images.'  # Example: 'Reviewed every indexed photo; recorded mixed scenes and exclusions.'
DISTINCT_NEAR_PAIR_REASONS = {}  # {'pairId from CSV': 'Different leaf shape and camera scene'}
FRESH_COLLECTION_NOTES = 'These 32 OOD photos were already reported in V4.1/V4.1.1; V4.2.0 reuse is exploratory, not independent.'

if not NEW_OOD_SCOPE_REVIEWED or len(SCOPE_REVIEW_NOTES.strip()) < 20:
    raise RuntimeError('Inspect every new OOD contact-sheet entry and record scope review first.')
flag_ids = {pair['pairId'] for pair in near_pairs}
if set(DISTINCT_NEAR_PAIR_REASONS) != flag_ids or any(
        len(reason.strip()) < 12 for reason in DISTINCT_NEAR_PAIR_REASONS.values()):
    raise RuntimeError('Review each possible near-duplicate pair; exclude true repeats and start a new run.')
verify_all_records_unchanged(all_records)
INPUT_PATH = RUN_DIR / 'evaluation_inputs.json'
write_json_once(INPUT_PATH, {'trainingCompleteSha256': COMPLETE_SHA256,
    'trainingManifestSha256': TRAIN_MANIFEST_SHA256,
    'referenceV411EvaluationInputsSha256': REFERENCE_V411_INPUTS_SHA256,
    'modelVersion': MODEL_VERSION,
    'armModelSha256': {a: arm_infos[a]['modelSha256'] for a in ARMS},
    'mango': manifest['images'], 'backgroundSources': background_rows,
    'oldOod': old_rows, 'newOod': new_rows, 'oldOodSplitMethod': old_split_method,
    'newOodTotalUploaded': len(new_all_paths), 'newOodRetained': len(new_rows),
    'oldOodEvaluationStatus': 'exploratory; previously used in model development',
    'newOodEvaluationStatus': 'reused report-only in V4.2.0; V4.1/V4.1.1 use recorded',
    'freshCollectionNotes': FRESH_COLLECTION_NOTES, 'scopeReviewNotes': SCOPE_REVIEW_NOTES,
    'distinctNearPairReasons': DISTINCT_NEAR_PAIR_REASONS,
    'exclusionReasons': EXCLUSION_REASONS, 'counts': counts,
    'reviewedAt': datetime.now(timezone.utc).isoformat()})
INPUT_SHA256 = sha256_file(INPUT_PATH)
val_records = [r for r in mango_records if r['subset'] == 'val']
test_records = [r for r in mango_records if r['subset'] == 'test']
tune_records = [r for r in old_rows if r['subset'] == 'ood_tune']
old_report_records = [r for r in old_rows if r['subset'] == 'ood_report']
new_report_records = new_rows
if min(map(len, [val_records, test_records, tune_records, old_report_records, new_report_records])) < 1:
    raise ValueError('Every validation, test, and OOD subset must contain images.')
print('Input review frozen:', INPUT_PATH, INPUT_SHA256)

Input review frozen: /content/drive/MyDrive/mangoleaf_experiments/v4.2.0/threshold_fitting/mangofruit/v4-2-0-mangofruit-thresholds-20260928T193804Z-fc01d0f7/evaluation_inputs.json 95d9d46a4f01462853ee147273dd5da03124e00ae0a811673a0934db102560f0


### Check frozen evaluation input inventory

Confirm `evaluation_inputs.json` binds only this organ's mango photos, old OOD tune/report, and reused new OOD report. No test or report inference has run yet.


## Step 4: Fit and freeze this organ's gate

**Reads only:** Mango validation and old OOD tune. Keep confidence 0.70; search margin and entropy from 0 to 1 in 0.01 increments. Require at least 95% validation mango retention. Maximize old OOD tune rejection, then favor higher retention, lower margin, and higher entropy. **Saves:** `selected_thresholds.json`, `arms/combined/threshold_search.csv`, full-precision tuning predictions. No held-out inference occurs here.


In [13]:
SELECTION_PATH = RUN_DIR / 'selected_thresholds.json'
if SELECTION_PATH.exists() or (RUN_DIR / 'evaluation_started.json').exists():
    raise RuntimeError('This run already fitted or evaluated. Preserve it; start a new run to change settings.')
verify_frozen_inputs()
selections = {}
search_rows = {}
tuning_rows = {}
for arm in ARMS:
    model = load_arm(arm)
    val_probabilities = predict_records(model, val_records)
    reference = np.load(arm_infos[arm]['probPath'], allow_pickle=False)
    if reference.shape != (len(val_records), len(CLASS_NAMES)):
        raise ValueError('Saved validation probability order/count differs: ' + arm)
    np.testing.assert_allclose(val_probabilities, reference, rtol=1e-4, atol=1e-4,
        err_msg='Saved validation predictions differ from this model or image loader: ' + arm)
    tune_probabilities = predict_records(model, tune_records)
    val_stats = probability_stats(val_probabilities.tolist())
    tune_stats = probability_stats(tune_probabilities.tolist())
    chosen, search = select_thresholds(val_stats, tune_stats, GRID,
                                        CONFIDENCE_THRESHOLD, MIN_ID_RETENTION)
    outcome = {'status': 'feasible' if chosen else 'infeasible', 'thresholds': chosen,
               'modelSha256': arm_infos[arm]['modelSha256'],
               'resumeInfoSha256': arm_infos[arm]['infoSha256'],
               'validationProbabilitiesSha256': arm_infos[arm]['validationProbabilitiesSha256'],
               'temperature': arm_infos[arm]['temperature'],
               'confidenceOnlyValidation': summarize(scored_rows(val_records, val_stats, CONFIDENCE_ONLY)),
               'confidenceOnlyOldOodTune': summarize(scored_rows(tune_records, tune_stats, CONFIDENCE_ONLY))}
    if chosen:
        outcome['validation'] = summarize(scored_rows(val_records, val_stats, chosen))
        outcome['oldOodTune'] = summarize(scored_rows(tune_records, tune_stats, chosen))
    selections[arm] = outcome
    search_rows[arm] = search
    tuning_rows[arm] = scored_rows(val_records, val_stats, chosen or CONFIDENCE_ONLY) +                        scored_rows(tune_records, tune_stats, chosen or CONFIDENCE_ONLY)
    print(arm, outcome['status'], chosen,
          'validation accepted:', outcome.get('validation', outcome['confidenceOnlyValidation'])['accepted'],
          'old OOD tune rejected:', outcome.get('oldOodTune', outcome['confidenceOnlyOldOodTune'])['rejected'])
    del model
    gc.collect()

verify_frozen_inputs()
selection = {'modelVersion': MODEL_VERSION, 'classes': CLASS_NAMES, 'arms': selections,
    'armOrder': ARMS, 'confidenceOnlyReference': CONFIDENCE_ONLY,
    'minimumValidationRetention': MIN_ID_RETENTION, 'grid': GRID,
    'objective': 'maximum old OOD tune rejection with minimum mango validation retention',
    'tieBreak': 'maximum validation retention, minimum margin, maximum entropy',
    'trainingCompleteSha256': COMPLETE_SHA256, 'evaluationInputsSha256': INPUT_SHA256,
    'newOodUsedForFitting': False, 'testOrReportPredictionsBeforeSelection': False,
    'runtime': RUNTIME, 'preprocessing': PREPROCESSING,
    'selectedAt': datetime.now(timezone.utc).isoformat()}
write_json_once(SELECTION_PATH, selection)
SELECTION_SHA256 = sha256_file(SELECTION_PATH)
for arm in ARMS:
    arm_dir = RUN_DIR / 'arms' / arm
    arm_dir.mkdir(parents=True, exist_ok=False)
    write_csv_once(arm_dir / 'threshold_search.csv', search_rows[arm])
    write_csv_once(arm_dir / 'tuning_predictions.csv', tuning_rows[arm])
print('ORGAN GATE FROZEN:', SELECTION_PATH)
print('Selection SHA-256:', SELECTION_SHA256)

combined infeasible None validation accepted: 77 old OOD tune rejected: 16
ORGAN GATE FROZEN: /content/drive/MyDrive/mangoleaf_experiments/v4.2.0/threshold_fitting/mangofruit/v4-2-0-mangofruit-thresholds-20260928T193804Z-fc01d0f7/selected_thresholds.json
Selection SHA-256: 8aee54a3d019a3c0e9b6d711aa0cce09e65ba92b6fad00df13751566cd18c33e


### Check gate selection

If status is `infeasible`, no fitted candidate can be exported; confidence-only remains a report reference. A margin threshold of 0 is a valid result and disables margin-based rejection.


## Step 5: Evaluate after gate freeze

**Reads:** Frozen gate and held-out mango test, old OOD report, reused new OOD report. **Saves:** `evaluation_started.json` before inference; then `evaluation.json`, raw per-class metrics, and full-precision per-image CSVs. Do not adjust thresholds from these results. Reused test/OOD numbers are exploratory, not fresh field validation.


In [14]:
if (RUN_DIR / 'evaluation_started.json').exists():
    raise RuntimeError('Evaluation already began. Keep this run intact; start another run if needed.')
frozen = read_frozen_selection(SELECTION_PATH, SELECTION_SHA256)
if (frozen['armOrder'] != ARMS or frozen['evaluationInputsSha256'] != INPUT_SHA256 or
        frozen['trainingCompleteSha256'] != COMPLETE_SHA256 or
        frozen['newOodUsedForFitting'] is not False):
    raise RuntimeError('Frozen organ gate does not match audited inputs.')
verify_frozen_inputs()
write_json_once(RUN_DIR / 'evaluation_started.json', {
    'selectionSha256': SELECTION_SHA256, 'inputSha256': INPUT_SHA256,
    'startedAt': datetime.now(timezone.utc).isoformat()})

from sklearn.metrics import classification_report, confusion_matrix
evaluations = {}
case_rows = {}
for arm in ARMS:
    selected = frozen['arms'][arm]
    if (selected['modelSha256'] != arm_infos[arm]['modelSha256'] or
            selected['resumeInfoSha256'] != arm_infos[arm]['infoSha256']):
        raise RuntimeError('Frozen gate belongs to different weights: ' + arm)
    model = load_arm(arm)
    test_probabilities = predict_records(model, test_records)
    old_report_probabilities = predict_records(model, old_report_records)
    new_report_probabilities = predict_records(model, new_report_records)
    stats_by_subset = {
        'mango_test': probability_stats(test_probabilities.tolist()),
        'old_ood_report': probability_stats(old_report_probabilities.tolist()),
        'new_ood_report': probability_stats(new_report_probabilities.tolist())}
    records_by_subset = {'mango_test': test_records, 'old_ood_report': old_report_records,
                         'new_ood_report': new_report_records}
    references = {'confidence_only': CONFIDENCE_ONLY}
    if selected['status'] == 'feasible':
        references['selected'] = selected['thresholds']
    arm_results = {}
    for gate_name, thresholds in references.items():
        reports = {}
        for subset, records_subset in records_by_subset.items():
            scored = scored_rows(records_subset, stats_by_subset[subset], thresholds)
            reports[subset] = summarize(scored)
            write_csv_once(RUN_DIR / 'arms' / arm / f'{gate_name}_{subset}_predictions.csv', scored)
            case_rows[(arm, gate_name, subset)] = scored
        arm_results[gate_name] = reports
    y_true = np.asarray([CLASS_NAMES.index(r['label']) for r in test_records])
    y_pred = test_probabilities.argmax(axis=1)
    raw_report = classification_report(y_true, y_pred, labels=list(range(len(CLASS_NAMES))),
        target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    raw_confusion = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))
    write_csv_once(RUN_DIR / 'arms' / arm / 'raw_per_class_metrics.csv',
                   [{'class': label, **raw_report[label]} for label in CLASS_NAMES])
    evaluations[arm] = {'status': selected['status'], 'thresholds': selected['thresholds'],
        'modelSha256': selected['modelSha256'], 'temperature': selected['temperature'],
        'rawClassificationReport': raw_report, 'rawConfusionMatrix': raw_confusion.tolist(),
        'calibratedTestCrossEntropy': float(-np.log(np.maximum(
            test_probabilities[np.arange(len(y_true)), y_true], 1e-7)).mean()),
        'gatedReports': arm_results}
    print(arm, 'raw test accuracy:', raw_report['accuracy'])
    for gate_name, reports in arm_results.items():
        print(' ', gate_name, 'wrong accepted:', reports['mango_test']['acceptedWrong'],
              'mango rejected:', reports['mango_test']['rejected'],
              'old OOD accepted:', reports['old_ood_report']['accepted'], '/', reports['old_ood_report']['total'],
              'new OOD accepted:', reports['new_ood_report']['accepted'], '/', reports['new_ood_report']['total'])
    del model
    gc.collect()

verify_frozen_inputs()
read_frozen_selection(SELECTION_PATH, SELECTION_SHA256)
evaluation = {'arms': evaluations, 'selectionSha256': SELECTION_SHA256,
    'evaluationInputsSha256': INPUT_SHA256,
    'oldTestAndOodStatus': 'exploratory; used in earlier model development',
    'newOodStatus': 'previously evaluated in V4.1/V4.1.1; reused report-only after gate freeze',
    'newOodCollectionNotes': FRESH_COLLECTION_NOTES,
    'newOodTotalUploaded': len(new_all_paths), 'newOodRetained': len(new_report_records),
    'oldOodTuneCount': len(tune_records), 'oldOodReportCount': len(old_report_records),
    'browserParityMeasured': False, 'freshLabeledMangoFieldAccuracyMeasured': False,
    'finishedAt': datetime.now(timezone.utc).isoformat()}
write_json_once(RUN_DIR / 'evaluation.json', evaluation)
EVALUATION_SHA256 = sha256_file(RUN_DIR / 'evaluation.json')
print('Evaluation saved:', RUN_DIR / 'evaluation.json')

combined raw test accuracy: 0.8958333333333334
  confidence_only wrong accepted: 0 mango rejected: 24 old OOD accepted: 64 / 84 new OOD accepted: 23 / 32
Evaluation saved: /content/drive/MyDrive/mangoleaf_experiments/v4.2.0/threshold_fitting/mangofruit/v4-2-0-mangofruit-thresholds-20260928T193804Z-fc01d0f7/evaluation.json


### Check evaluation output

`evaluation.json` separates raw accuracy, accepted correct/wrong mango photos, mango rejections, and OOD false accepts for frozen and confidence-only gates. Per-image CSVs include full class probabilities and rejection reasons. These results cannot be used to re-fit the same reported gate.


## Step 6: Inspect one saved prediction

Choose a subset and zero-based index. This reads Step 5's saved decision and prints all class percentages without rerunning inference. Use it for error analysis only.


In [30]:
import random

DEBUG_ARM = 'combined'  # Only V4.2.0 arm for this organ.
DEBUG_SUBSET = 'old_ood_report'  # mango_test, old_ood_report, new_ood_report
DEBUG_IMAGE_INDEX = random.randrange(0,50)
DEBUG_GATE = 'confidence_only'  # selected or confidence_only

read_frozen_selection(SELECTION_PATH, SELECTION_SHA256)
if DEBUG_ARM not in ARMS or DEBUG_SUBSET not in records_by_subset:
    raise ValueError('Choose a saved arm and evaluated subset.')
if (DEBUG_ARM, DEBUG_GATE, DEBUG_SUBSET) not in case_rows:
    raise ValueError('This arm has no feasible selected gate; choose confidence_only.')
preview = case_rows[(DEBUG_ARM, DEBUG_GATE, DEBUG_SUBSET)]
if not isinstance(DEBUG_IMAGE_INDEX, int) or not 0 <= DEBUG_IMAGE_INDEX < len(preview):
    raise ValueError('DEBUG_IMAGE_INDEX is outside this saved subset.')
row = preview[DEBUG_IMAGE_INDEX]
print('Image:', row['relativePath'], '| actual:', row['label'])
print('Decision:', row['returnedClass'], '| rejection reasons:', row['rejectionReasons'] or 'none')
print_class_probabilities([row['probability_' + label] for label in CLASS_NAMES],
                          DEBUG_ARM + ' / ' + DEBUG_SUBSET)

Image: random_img_17.jpg | actual: OOD
Decision: uncertain | rejection reasons: confidence

combined / old_ood_report
Mango Scab                48.29%
Bacterial Black Spot      37.05%
Healthy Fruit             14.67%


## Step 7: Package Mango Fruit candidate and evidence

**Reads:** Frozen selection/evaluation and verified training TensorFlow.js export. **Saves:** A feasible `candidate_models/combined/` with unchanged graph/weights and fitted `model_config.json`, candidate hashes/ZIP, report, evidence ZIP. If infeasible, no candidate folder is created. The app remains unchanged. Future app integration would copy the entire candidate into `public/models/mangofruit/` and separately implement picker, cache, and browser checks.


In [31]:
read_frozen_selection(SELECTION_PATH, SELECTION_SHA256)
if sha256_file(RUN_DIR / 'evaluation.json') != EVALUATION_SHA256:
    raise RuntimeError('Evaluation changed before packaging.')
verify_frozen_inputs()
candidate_root = RUN_DIR / 'candidate_models'
candidate_root.mkdir(exist_ok=False)
candidate_paths = {}
for arm in ARMS:
    selected = frozen['arms'][arm]
    if selected['status'] != 'feasible':
        continue
    source = TRAINING_RUN / 'arms' / arm / 'tfjs_model'
    model_files = verify_tfjs_export(source)
    for relative in model_files:
        trained_relative = f'arms/{arm}/tfjs_model/{relative}'
        if complete['artifacts'][trained_relative] != sha256_file(inside(source, relative)):
            raise RuntimeError('TFJS export changed after training: ' + trained_relative)
    candidate_dir = candidate_root / arm
    candidate_dir.mkdir(exist_ok=False)
    for relative in model_files:
        destination = inside(candidate_dir, relative)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(inside(source, relative), destination)
    config = {'modelVersion': MODEL_VERSION, 'experimentVersion': 'v4.2.0', 'organ': ORGAN, 'arm': arm,
        'classes': CLASS_NAMES, 'inputSize': list(IMAGE_SIZE),
        'preprocessing': PREPROCESSING, 'temperature': selected['temperature'],
        'temperatureFoldedIntoWeights': True, 'thresholds': selected['thresholds'],
        'status': 'experimental candidate; not adopted or deployed',
        'modelSha256': selected['modelSha256'], 'selectionSha256': SELECTION_SHA256,
        'evaluationSha256': EVALUATION_SHA256, 'trainingCompleteSha256': COMPLETE_SHA256,
        'browserParityMeasured': False}
    write_json_once(candidate_dir / 'model_config.json', config)
    candidate_zip = RUN_DIR / f'{RUN_DIR.name}-{arm}-candidate_model.zip'
    with zipfile.ZipFile(candidate_zip, 'x', compression=zipfile.ZIP_DEFLATED) as bundle:
        for path in sorted(candidate_dir.rglob('*')):
            if path.is_file():
                bundle.write(path, path.relative_to(candidate_dir).as_posix())
    candidate_paths[arm] = str(candidate_zip)
write_json_once(RUN_DIR / 'candidate_hashes.json', {
    arm: {p.relative_to(candidate_root / arm).as_posix(): sha256_file(p)
          for p in sorted((candidate_root / arm).rglob('*')) if p.is_file()}
    for arm in candidate_paths})

report_lines = [
    '# Mango Fruit V4.2.0: single-model threshold experiment', '',
    'Experimental candidate. No app model was changed or deployed.',
    f'Training run: `{TRAINING_RUN}`',
    f'Selection SHA-256: `{SELECTION_SHA256}`',
    f'Evaluation SHA-256: `{EVALUATION_SHA256}`', '',
    'Gate was fitted on mango validation and old OOD tune, then frozen before test and report inference.',
    'Old mango test and old OOD report results are exploratory because those collections were used in earlier development.',
    'The 32 V4.1 new OOD photos are reused as a report-only collection here; they are not fresh independent evidence. Collection history appears below.', '',
    f'Counts: validation {len(val_records)}, mango test {len(test_records)}, old OOD tune {len(tune_records)}, old OOD report {len(old_report_records)}, new OOD uploaded {len(new_all_paths)}, new OOD retained {len(new_report_records)}.',
    f'New OOD scope review: {SCOPE_REVIEW_NOTES}',
    f'Collection history: {FRESH_COLLECTION_NOTES}', '',
    '| Model | Gate | Test correct accepted | Test wrong accepted | Mango rejected | Old OOD accepted | New OOD accepted |',
    '| --- | --- | ---: | ---: | ---: | ---: | ---: |']
for arm in ARMS:
    result = evaluations[arm]
    gate = 'selected' if result['status'] == 'feasible' else 'confidence_only (infeasible)'
    reports = result['gatedReports']['selected' if result['status'] == 'feasible' else 'confidence_only']
    mango, old_ood, new_ood = (reports[k] for k in ('mango_test', 'old_ood_report', 'new_ood_report'))
    report_lines.append(f'| {arm} | {gate} | {mango["acceptedCorrect"]}/{mango["total"]} | '
        f'{mango["acceptedWrong"]} | {mango["rejected"]} | '
        f'{old_ood["accepted"]}/{old_ood["total"]} | {new_ood["accepted"]}/{new_ood["total"]} |')
report_lines += ['',
    'Confidence-only and selected-gate metrics, 3-class per-class precision/recall/F1, class confusion, and full-precision per-image probabilities are in `evaluation.json` and `arms/` CSV files.',
    'Full-precision per-image CSVs record every decision and rejection reason.',
    'Validation retention measures accepted images, including accepted wrong diagnoses; it is not accuracy.',
    'Background replacement and local light do not train an explicit non-mango class. No field mango accuracy or TensorFlow.js browser parity was measured here.',
    'If reused OOD scores guide model choice, collect another untouched OOD set for final independent evaluation.',
    'Expert-labeled mango field images are needed to evaluate harsh-light disease diagnosis.']
report_path = RUN_DIR / 'REPORT.md'
with report_path.open('x', encoding='utf-8') as handle:
    handle.write('\n'.join(report_lines) + '\n')

evidence_zip = RUN_DIR / f'{RUN_DIR.name}-evidence.zip'
with zipfile.ZipFile(evidence_zip, 'x', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RUN_DIR.rglob('*')):
        if path.is_file() and path.suffix != '.zip' and not path.is_relative_to(candidate_root):
            bundle.write(path, path.relative_to(RUN_DIR).as_posix())
    for name in ('training_complete.json', 'input_manifest.json'):
        bundle.write(TRAINING_RUN / name, 'training/' + name)
    for arm in ARMS:
        bundle.write(TRAINING_RUN / 'arms' / arm / 'resume_info.json',
                     'training/arms/' + arm + '/resume_info.json')
print('\n'.join(report_lines))
print('Threshold run for post-threshold figures:', RUN_DIR)
print('Evidence ZIP:', evidence_zip)
for arm, path in candidate_paths.items():
    print('Experimental candidate ZIP:', arm, path)

# Mango Fruit V4.2.0: single-model threshold experiment

Experimental candidate. No app model was changed or deployed.
Training run: `/content/drive/MyDrive/mangoleaf_experiments/v4.2.0/training/mangofruit/v4-2-0-mangofruit-training-20260928T164258Z-a4e1abae`
Selection SHA-256: `8aee54a3d019a3c0e9b6d711aa0cce09e65ba92b6fad00df13751566cd18c33e`
Evaluation SHA-256: `8215f0bdd5ada95860ba1e12d80bdf41131576a6acc48de013d0cabb8950c3cc`

Gate was fitted on mango validation and old OOD tune, then frozen before test and report inference.
Old mango test and old OOD report results are exploratory because those collections were used in earlier development.
The 32 V4.1 new OOD photos are reused as a report-only collection here; they are not fresh independent evidence. Collection history appears below.

Counts: validation 94, mango test 96, old OOD tune 84, old OOD report 84, new OOD uploaded 32, new OOD retained 32.
New OOD scope review: Reviewed all the images.
Collection history: These 32 OOD phot

### Final file map

- `threshold_fitting/mangofruit/<run>/selected_thresholds.json` and `evaluation.json`: frozen gate and held-out report.
- `arms/combined/threshold_search.csv`, `raw_per_class_metrics.csv`, and `selected_*_predictions.csv`: figure inputs in matching training notebook. If infeasible, use confidence-only CSVs and do not claim a fitted candidate.
- `candidate_models/combined/` and `candidate_hashes.json`: full experimental model export and file hashes if feasible. This is not `public/models/mangofruit/`.
- `audit/`, `REPORT.md`, and evidence ZIP: reviewed input scope and provenance. Keep full Drive runs because ZIP omits large model weights.

Paste printed threshold run path into matching training notebook's post-threshold figure section. This clean template contains no measurements.


## References

- [Keras transfer learning](https://keras.io/guides/transfer_learning/) and [Keras image loading](https://keras.io/api/data_loading/image/) document saved model and image decoding.
- [Scikit-learn data leakage guidance](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage) explains why fitting and report sets stay separate.
- [TensorFlow.js SavedModel conversion](https://www.tensorflow.org/js/tutorials/conversion/import_saved_model) documents browser export format.

Actual model quality requires completed Colab runs; notebook syntax checks alone cannot establish it.
